# Fundamentals of Data Science - Network Science Lab

# Lab Session 05 - Introduction to Random Networks

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

## Network Descriptives

Network descriptives can be grouped into global descriptives, which capture features relating to the whole network, and local metrics, which describe features for a single node.

### Clustering Coefficient

The clustering coefficient measures the extent to which nodes tend to form groups within a network.

Since the simplest structural group is a triangle, the global clustering coefficient is defined as the proportion of triangles to connected triples, meaning that
$$C = \frac{\text{Number of Triangles}}{\text{Number of Connected Triples}}$$

Using the $l^{th}$ power of the adjacency matrix, which indicates the number of distinct paths of length $l$ from node $i$ to node $j$, the global clustering coefficient can also be formulated as
$$C = \frac{tr(A^3)}{\sum_{i \neq j} A_{ij}^2}$$

By contrast, the local clustering of a node represents the density of the subgraph formed by its neighbours, meaning that
$$c_i = \frac{2m_i}{k_i(k_i - 1)}, \text{with $c_i = 0$ if $k_i \leq 1$.}$$

In particular, it is possible to express the global clustering coefficient of a network as the average local clustering coefficient of its nodes, resulting in
$$C = \frac{1}{|V|}\sum_{i=1}^{|V|} c_i$$

In [ ]:
base_dir = Path.cwd()
vertex_output_size = 30
edges = [(0,1), (0,2), (1,2), (2,3), (3,4), (3,5), (4,5)]
g_py = ig.Graph(edges=edges, directed=False)

# Plot the network.
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_py,
    target=ax,
    vertex_label=list(range(0, g_py.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2,
)

In [ ]:
# Get the adjacency matrix of the network.
A_g_py = np.array(g_py.get_adjacency().data, dtype=float)

# Compute A^2 and A^3.
A2_g_py = A_g_py @ A_g_py
print("The squared adjacency \n", A2_g_py)

A3_g_py = A2_g_py @ A_g_py  # Alternatively, A3_g_py = A_g_py @ A_g_py @ A_g_py.
print("The power 3 adjacency \n", A3_g_py)

In [ ]:
# Compute the global clustering coefficient manually.
numerator = np.trace(A3_g_py)
denominator = np.sum(A2_g_py) - np.trace(A2_g_py)
C_manual = numerator / denominator
print("Manual Clustering Coefficient: ", C_manual)

# Compute the global clustering coefficient using the transitivity metric.
print("Global Clustering Coefficient: ",g_py.transitivity_undirected())

# Compute the local clustering coefficient.
print("Local Clustering Coefficient: ",g_py.transitivity_local_undirected())
print("Average Local Clustering Coefficient: ",g_py.transitivity_avglocal_undirected())
print("Check: ",np.mean(g_py.transitivity_avglocal_undirected()))

### Node Centrality

Centrality metrics come in handy to identify structurally important nodes in a network.

Generally speaking, node centrality can be evaluated according to one of the following metrics:
- **Closeness:** This metric measures how close a node is to the other nodes based on shortest paths, meaning that
  $$ Closeness(i) = \frac{|V| - 1}{\sum_{j \in V \setminus \{i\}} d_{i, j}}$$
- **Betweenness:** This metric measures the extent to which a node can act as a bridge along the shortest paths between pairs of nodes.

  In fact, assuming $\sigma_{st}$ denotes the number of shortest paths between nodes $s$ and $t$, with $\sigma_{st}(i)$ indicating the number of paths passing through node $i$, the betweenness centrality of a node can be defined as
  $$Betweenness(i) = \sum_{s \neq i \neq t} \frac{\sigma_{st}(i)}{\sigma_{st}}$$

In [ ]:
# Study metrics for a specific node.
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
  g_py,
  target=ax,
  vertex_label=list(range(0, g_py.vcount())),
  vertex_size=vertex_output_size,
  vertex_color=["lightblue", "lightblue", "yellow", "lightblue", "lightblue", "lightblue"],
  edge_width=2,
)

In [ ]:
# Compute centrality metrics for a specific node.
g_py.closeness(2) # Closeness centrality, which can be normalized by specifying normalized=True.
g_py.betweenness(2) # Betweenness centrality.

## The Erdős-Rényi Model

Random graph models come in handy to construct ensembles, which represent probability distributions over possible network configurations.

In fact, one can assess the statistical significance of a property observed in a real network by comparing it against synthetic random graphs.

The Erdős-Rényi model is an ensemble $G(n, p)$ that fixes the number of nodes $n$ and the independent probability $p$ of having an edge between any two nodes.

In [ ]:
random.seed(10) # For reproducibility.

# Create four networks using the Erdős-Rényi model, choosing n = 25 and p = 0.1.
er1 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er2 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er3 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)
er4 = ig.Graph.Erdos_Renyi(n = 25, p = 0.1)

# Plot the generated networks.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    er1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    er2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    er3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    er4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()

Since edges may or may not be added through independent and identically distributed Bernoulli trials, it is possible to discover that the edges in an Erdős-Rényi network follow a binomial distribution defined by
$$\mathbb{P}(m) = p^m(1 - p)^{\binom{n}{2} - m} \Rightarrow \langle m \rangle = \binom{n}{2}p$$

This phenomenon implies that the degree distribution of an Erdős-Rényi network is also binomial, with
$$p_k = \binom{n - 1}{k}p^k(1 - p)^{n - 1 - k} \Rightarrow \langle k \rangle = (n - 1)p$$

However, for many real networks, where $n \to +\infty$, the degree distribution tends to be approximated with a Poisson distribution with rate $\lambda = \langle k \rangle$, resulting in
$$p_k = \frac{e^{-\langle k \rangle}\langle k \rangle^k}{k!}$$

However, since the binomial and Poisson distributions lack heavy tails, the Erdős-Rényi model fails to reproduce the heavy-tailed degree distributions observed in real networks.

In [ ]:
def degree_distribution(graph, mode="all"):
  # This function computes the degree distribution of a (directed) graph.
  degrees = graph.degree(mode=mode)
  degree_frequency = pd.Series(degrees).value_counts()
  degree_frequency = degree_frequency.sort_index()
  full_range = range(min(graph.degree()+[1]), max(graph.degree())+1)
  degree_frequency = degree_frequency.reindex(full_range, fill_value=0)

  degree_distribution = degree_frequency / degree_frequency.sum()
  return degree_distribution

# Analyse the degree distribution of an Erdős-Rényi random network.
g_er = ig.Graph.Erdos_Renyi(n = 100, p = 0.1)
g_er.summary()

fig, ax = plt.subplots(figsize=(5, 4))

# Plot the distribution using a frequency scatterplot.
ax.scatter(degree_distribution(g_er).index,
degree_distribution(g_er).values, color = "red")
ax.set_title("Frequency of Elements (Scatter Plot)")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.set_box_aspect(1)

In [ ]:
random.seed(10) # For reproducibility.

# Extract information about the degree and edge distributions of 100 Erdős-Rényi random networks with n = 100 and p = 0.1.
er_degree_values = []
er_m = []
for i in range(0,100):
  tmp_er = ig.Graph.Erdos_Renyi(n = 100, p = 0.1)
  er_degree_values.append(tmp_er.degree())
  er_m.append(tmp_er.ecount())

# Plot the resulting distributions.
fig, ax = plt.subplots(1, 2, figsize=(9, 4))#

# Degree distribution.
ax[0].hist(np.array(er_degree_values).flat, bins = 25,
           color = "red", edgecolor = "white")
ax[0].axvline(x=99*0.1, color='black', linestyle='--',
              linewidth=2, label='E(k)')
ax[0].set_title("Degree Distribution - 100 ER Networks")
ax[0].set_xlabel("Degree")
ax[0].set_ylabel("Frequency")
ax[0].set_box_aspect(1)
ax[0].legend()

# Edge distribution.
ax[1].hist(er_m, bins = 15,
           color = "red", edgecolor = "white")
ax[1].axvline(x=0.5*100*99*0.1, color='black', linestyle='--',
              linewidth=2, label='E(m)')
ax[1].set_title("Number of Links - 100 ER Networks")
ax[1].set_xlabel("Number of Links")
ax[1].set_ylabel("Frequency")
ax[1].set_box_aspect(1)
ax[1].legend()

plt.show()

While the Erdős-Rényi model fails to capture real degree distributions, it automatically encodes the small-world property.

In fact, it is possible to show inductively that the average number of nodes located at distance $s$ from a source node is equal to $\langle k \rangle^s$.

Therefore, assuming an Erdős-Rényi network has diameter $d$, it will be the case that
$$ \langle k \rangle^d \approx n \Rightarrow d = \frac{\ln(n)}{\ln(\langle k \rangle)}$$

The size of the largest connected component in an Erdős-Rényi network depends on the probability of having an edge between any two nodes.

More specifically, when $p = 0$, the network consists of a set of small isolated components that do not depend on the size of the network.

By contrast, if $p = 1$, the network acts a single $n$-clique that does depend on the size of the network.

In [ ]:
def sample_gnk(n, k, directed=False, loops=False):
  # Use an alternative parametrization for the Erdős-Rényi model.
  pk = k/(n-1)
  g = ig.Graph.Erdos_Renyi(n = n, p = pk, directed = directed, loops = loops)
  return g

In [ ]:
random.seed(10) # For reproducibility.

results = []

for k in np.arange(0,3.1,0.25):
  for r in range(10):
    # Store information about the largest connected component of each sampled network.
    tmp_g = sample_gnk(n = 100, k = k)
    rel_size_lcc = tmp_g.components().giant().vcount()/tmp_g.vcount()
    p = round(k/(99),ndigits=2)

    results.append({
        "k":k,
        "p": p,
        "replicate": r,
        "rel_size_lcc": rel_size_lcc
    })


# Store the results in a dataframe.
df_results = pd.DataFrame(results)
print(df_results.head())

In [ ]:
df_agg = df_results.groupby("k").agg({
    "rel_size_lcc": ["mean", "std"]
}).reset_index()

print(df_agg)

In [ ]:
# Plot the relationship between the expected degree and the relative size of the largest connected component.
fig, ax = plt.subplots(figsize=(5, 4))

ax.errorbar(df_agg["k"], df_agg["rel_size_lcc"]["mean"],
             yerr=df_agg["rel_size_lcc"]["std"],
             fmt='-o', label="Average S")
ax.axhline(y=0.25, color='red', linestyle='--',
              linewidth=1),
ax.axhline(y=0.8, color='red', linestyle='--',
              linewidth=1),
ax.set_xlabel("<k> (Expected Degree)")
ax.set_ylabel("S (Relative Size of LCC)")
ax.legend()

plt.show()

Overall, for an Erdős-Rényi network, it is possible to prove that, if $S$ is the proportion of nodes belonging to the largest connected component, the following relation holds:
$$S = 1 - e^{-S\langle k \rangle} \Rightarrow \langle k \rangle = \frac{\ln(1 - S)}{S}$$

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.errorbar(df_agg["k"], df_agg["rel_size_lcc"]["mean"],
             yerr=df_agg["rel_size_lcc"]["std"],
             fmt='-o', label="Average S")
ax.plot(theoretical_k,theoretical_s, label="Theoretical", color="blue")
ax.set_xlabel("<k> (Expected Degree)")
ax.set_ylabel("S (Relative Size of LCC)")
ax.legend()

plt.show()

The Erdős-Rényi model can also be defined using a $G(n, m)$ ensemble that fixes the number of nodes $n$ and the number of edges $m$, making it useful to create reference systems in which edge counts must remain constant.

In [ ]:
random.seed(10) # For reproducibility.

# Create four networks using the Erdős-Rényi model, choosing n = 25 and a parameter m.
m = round(0.1*24*24/2)
er1 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er2 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er3 = ig.Graph.Erdos_Renyi(n = 25, m = m)
er4 = ig.Graph.Erdos_Renyi(n = 25, m = m)

# Plot the generated networks.
fig, ax = plt.subplots(2,2,figsize=(9, 8))

ig.plot(
    er1,
    target=ax[0,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,0].set_title("First Graph")

ig.plot(
    er2,
    target=ax[0,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[0,1].set_title("Second Graph")

ig.plot(
    er3,
    target=ax[1,0],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,0].set_title("Third Graph")

ig.plot(
    er4,
    target=ax[1,1],
    vertex_size=20,
    vertex_color="lightblue",
    edge_width=2,
)
ax[1,1].set_title("Fourth Graph")

plt.tight_layout()
plt.show()